### Create Bronze Tables for Historical Data

In [0]:
from pyspark.sql.functions import current_timestamp
import os

In [0]:


# list all kaggle csv files
files=dbutils.fs.ls("/Volumes/f1_warehouse/ingestion/raw_files/kaggle/")

for file in files:

    # remove the ext to get the table name
    table_name=os.path.splitext(file.name)[0]

    #read csv
    df=(spark.read.option("header", True).option("inferSchema", True).csv(file.path))

    # add metadata
    bronze_df=(df.withColumn("ingestion_timestamp", current_timestamp())
                    .withColumn("file_name", df["_metadata.file_path"]))

    # Write as a Delta table
    (
        bronze_df.write# save as df
        .format("delta")# cahgne to delta format
        .mode("overwrite")#if table exists overwrite
        .saveAsTable(f"f1_warehouse.bronze.kaggle_{table_name}")
    )

    print(f"Created bronze.kaggle_{table_name}")







### Data Checks

In [0]:
%sql
SHOW TABLES IN  f1_warehouse.bronze;

all tables are loaded into bronze